iEMG

sENG

sEMG (gestures 1-10)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os
import scipy
import sys

sys.path.append('../Exp6 - iENG vs sENG')
import shared_iENG_sENG
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D


def X_y_from_matfile(path, modality, session):
    X, y = [], []

    if modality=='EMG':
        bluetooth_id = 'E8DD80E550BB'
    elif modality=='ENG':
        bluetooth_id = 'E9AD0E7DCC2B'

    data_per_class_files = os.listdir(path+f'{modality}_{session}/{bluetooth_id}/raw/')

    for cls in data_per_class_files:
        input_path = path+f'{modality}_{session}/{bluetooth_id}/raw/{cls}/'
        files = os.listdir(input_path)

        mat = scipy.io.loadmat(input_path+files[0])
        x_tmp = mat['Data_Fea'].transpose(2, 0, 1)

        X.append(x_tmp.reshape(x_tmp.shape[0], x_tmp.shape[1]*x_tmp.shape[2]))
        y.append(mat['Data_Cls'].ravel())

    return X, y


from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist

def t_sne_compute_dist(sEMG, sENG, iENG, vis=False):

    # Stack all for t-SNE
    X_all = np.vstack([sEMG, sENG, iENG])
    labels = np.array([0]*sEMG.shape[0] + [1]*sENG.shape[0] + [2]*iENG.shape[0])

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    X_tsne = tsne.fit_transform(X_all)

    names = ['sEMG', 'sENG', 'iENG']
    # Visualize
    if vis:
        plt.figure(figsize=(8,6))
        colors = ['r', 'g', 'b']

        for i, c, name in zip(range(3), colors, names):
            plt.scatter(X_tsne[labels==i, 0], X_tsne[labels==i, 1], c=c, label=name, alpha=0.6, s=20)

        plt.legend()
        plt.title("t-SNE of three datasets")
        plt.show()

    # Compute prototypes (mean feature vector)
    proto_i = sEMG.mean(axis=0)
    proto_s1 = sENG.mean(axis=0)
    proto_s2 = iENG.mean(axis=0)

    # Stack prototypes
    prototypes = np.vstack([proto_i, proto_s1, proto_s2])

    # Compute pairwise distances
    dist_matrix = cdist(prototypes, prototypes, metric='euclidean')
    dist_matrix = pd.DataFrame(dist_matrix, index=names, columns=names)
    #print("Pairwise distances between prototypes:\n", dist_matrix)
    return dist_matrix


from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def t_sne_compute_dist_3d(sEMG, sENG, iENG):

    # Stack all for t-SNE
    X_all = np.vstack([sEMG, sENG, iENG])
    labels = np.array([0]*sEMG.shape[0] + [1]*sENG.shape[0] + [2]*iENG.shape[0])

    # 3D t-SNE
    tsne = TSNE(n_components=3, random_state=42)
    X_tsne = tsne.fit_transform(X_all)

    # 3D Visualization
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111, projection='3d')

    colors = ['r', 'g', 'b']
    names = ['sEMG', 'sENG', 'iENG']

    for i, c, name in zip(range(3), colors, names):
        ax.scatter(
            X_tsne[labels==i, 0],
            X_tsne[labels==i, 1],
            X_tsne[labels==i, 2],
            c=c, label=name, alpha=0.6, s=20
        )

    ax.set_title("3D t-SNE of three datasets")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_zlabel("t-SNE 3")
    ax.legend()
    plt.show()

    # Compute prototypes (mean feature vector)
    proto_i = sEMG.mean(axis=0)
    proto_s1 = sENG.mean(axis=0)
    proto_s2 = iENG.mean(axis=0)

    # Stack prototypes
    prototypes = np.vstack([proto_i, proto_s1, proto_s2])

    # Compute pairwise distances
    dist_matrix = cdist(prototypes, prototypes, metric='euclidean')
    dist_matrix = pd.DataFrame(dist_matrix, index=names, columns=names)

    return dist_matrix


# invasive ENG (Amputee)

In [2]:
def get_iENG(gesture_label):
    path = 'C:/Users/hml76/Desktop/Jupyter/Federated Prototype Learning/Dataset/Ours_cleaned/'
    fname = os.listdir(path)

    iENG_subjects = {
        "subject2": {"X": [], "Y": []},
        "subject3": {"X": [], "Y": []},
    }

    for f in fname:
        for i in range(10):
            X = np.array(pd.read_csv(path + f'{f}/rep{i}_X.csv'))
            y = np.array(pd.read_csv(path + f'{f}/rep{i}_Y.csv'))
            #indices = np.random.permutation(len(X)) # shuffle
            #X, y = X[indices], y[indices]

            y_int = np.argmax(y, axis=1)
            num_classes = np.max(y_int) + 1

            if "subject2" in f:
                iENG_subjects["subject2"]["X"].append(X)
                iENG_subjects["subject2"]["Y"].append(y_int)
            elif "subject3" in f:
                iENG_subjects["subject3"]["X"].append(X)
                iENG_subjects["subject3"]["Y"].append(y_int)

    final_session = 4
    y = iENG_subjects["subject2"]['Y'][2:final_session] + iENG_subjects["subject3"]['Y'][2:final_session]
    x = iENG_subjects["subject2"]['X'][2:final_session] + iENG_subjects["subject3"]['X'][2:final_session]

    y = np.concatenate(y, axis=0)
    x = np.concatenate(x, axis=0)
    x = x.reshape(x.shape[0], 16, 14)

    x = x[:, [0, 4, 8, 12], :][:, :, [2, 4]]   # (N, 4, 5)
    x = x.reshape(x.shape[0], -1)           # (N, 20)
    # others => [0, 2, 3, 4, 12] # ZC, WL, WAMP, MAV, MAVS

    #mask = (y == 0)
    #mask = (y >= 1) & (y <= 5)
    mask = (y == gesture_label)
    iENG_x_zero = x[mask]
    iENG_y_zero = y[mask]
    print(iENG_x_zero.shape, iENG_y_zero.shape)

    return iENG_x_zero, iENG_y_zero

# sEMG (Ninapro)

In [3]:
def get_sEMG():
    b_path = 'C:/Users/hml76/Desktop/Jupyter/Federated Prototype Learning/Dataset/DB4'

    X_lst, y_lst = [], []

    #for sub in range(10, 20):
    for sub in [1,4,5,7]:
        for rep in range(1,6):
            X = pd.read_csv(b_path+f'/Sub{sub}/Sub{sub}_Rep{rep}_data.csv')
            y = pd.read_csv(b_path+f'/Sub{sub}/Sub{sub}_Rep{rep}_label.csv')
            y = np.argmax(y, axis=1)
            mask = (y == 0)
            X = np.array(X).reshape(X.shape[0], 12, 11)
            X = X[:, [2,6,8,0], 0:2] #ninapro => [0, 1, 2, 3, 4]: MAV, WL, WAMP, ZC, MAVS
            X = X.reshape(X.shape[0], 8)
            X_lst.append(X[mask])
            y_lst.append(y[mask])

    sEMG_x_zero = np.concatenate(X_lst, axis=0)
    sEMG_y_zero = np.concatenate(y_lst, axis=0)

    print(sEMG_x_zero.shape, sEMG_y_zero.shape)
    return sEMG_x_zero, sEMG_y_zero


def get_sEMG_3DBs(gesture_label1, gesture_label2):
    b_path1 = 'C:/Users/hml76/Desktop/Jupyter/Federated Prototype Learning/Dataset/DB4'
    b_path2 = 'C:/Users/hml76/Desktop/Jupyter/Federated Prototype Learning/Dataset/DB5'
    b_path3 = 'C:/Users/hml76/Desktop/Jupyter/Federated Prototype Learning/Dataset/DB1'

    X_lst, y_lst = [], []

    #for sub in range(10, 20):
    for sub in [1,4]:
        for rep in range(1,4):
            X = pd.read_csv(b_path1+f'/Sub{sub}/Sub{sub}_Rep{rep}_data.csv')
            y = pd.read_csv(b_path1+f'/Sub{sub}/Sub{sub}_Rep{rep}_label.csv')
            y = np.argmax(y, axis=1)
            #mask = (y == 0)
            mask = (y >= gesture_label1) & (y <= gesture_label2)
            X = np.array(X).reshape(X.shape[0], 12, 11)
            X = X[:, [2,6,8,0], 0:2] #ninapro => [0, 1, 2, 3, 4]: MAV, WL, WAMP, ZC, MAVS
            X = X.reshape(X.shape[0], 8)
            X_lst.append(X[mask])
            y_lst.append(y[mask])

    for sub in [1,4]:
        for rep in range(1,4):
            X = pd.read_csv(b_path2+f'/Sub{sub}_serial/Sub{sub}_Rep{rep}_data.csv')
            y = pd.read_csv(b_path2+f'/Sub{sub}_serial/Sub{sub}_Rep{rep}_label.csv')
            y = np.argmax(y, axis=1)
            #mask = (y == 0)
            mask = (y >= gesture_label1) & (y <= gesture_label2)
            X = np.array(X).reshape(X.shape[0], 16, 11)
            X = X[:, [2,6,8,0], 0:2] #ninapro => [0, 1, 2, 3, 4]: MAV, WL, WAMP, ZC, MAVS
            X = X.reshape(X.shape[0], 8)
            X_lst.append(X[mask])
            y_lst.append(y[mask])

    for sub in [17,19]:
        for rep in range(1,4):
            X = pd.read_csv(b_path3+f'/Sub{sub}/Sub{sub}_Rep{rep}_data.csv')
            y = pd.read_csv(b_path3+f'/Sub{sub}/Sub{sub}_Rep{rep}_label.csv')
            y = np.argmax(y, axis=1)
            #mask = (y == 0)
            #mask = (y >= 1) & (y <= 10)
            mask = (y >= gesture_label1) & (y <= gesture_label2)
            X = np.array(X).reshape(X.shape[0], 10, 11)
            X = X[:, [2,6,8,0], 0:2] #ninapro => [0, 1, 2, 3, 4]: MAV, WL, WAMP, ZC, MAVS
            X = X.reshape(X.shape[0], 8)
            X_lst.append(X[mask])
            y_lst.append(y[mask])

    sEMG_x_zero = np.concatenate(X_lst, axis=0)
    sEMG_y_zero = np.concatenate(y_lst, axis=0)

    print(sEMG_x_zero.shape, sEMG_y_zero.shape)
    return sEMG_x_zero, sEMG_y_zero

# sENG (HM, BY, MJ)

In [4]:
def get_sENG(gesture_label):
    bluetooth_id = 'E9AD0E7DCC2B'
    base_path = 'C:/Users/hml76/PycharmProjects/MindForce/data/sENG_iENG/'
    X_sENG_sub1, y_sENG_sub1 = shared_iENG_sENG.return_X_y_get_from_matfile(path = base_path+f'Hunmin/{bluetooth_id}/raw/', num_session=[0,1,2,3,4], balance=True)
    X_sENG_sub2, y_sENG_sub2 = shared_iENG_sENG.return_X_y_get_from_matfile(path = base_path+f'Byeongchan/{bluetooth_id}/raw/', num_session=[0,1,2,3,4], balance=True)
    X_sENG_sub3, y_sENG_sub3 = shared_iENG_sENG.return_X_y_get_from_matfile(path = base_path+f'Minjeong/{bluetooth_id}/raw/', num_session=[0,1,2,3,4], balance=True)
    X_sENG, y_sENG = np.concatenate([X_sENG_sub1, X_sENG_sub2, X_sENG_sub3], axis=0), np.concatenate([y_sENG_sub1, y_sENG_sub2, y_sENG_sub3], axis=0)

    x, y = X_sENG, y_sENG

    #mask = (y == 0)
    #mask = (y >= 1) & (y <= 5)
    mask = (y == gesture_label)
    sENG_x_zero = x[mask]
    sENG_y_zero = y[mask]

    sENG_x_zero = sENG_x_zero[:, :, [2, 4]]
    sENG_x_zero = sENG_x_zero.reshape(sENG_x_zero.shape[0], 8)

    print(sENG_x_zero.shape, sENG_y_zero.shape)
    return sENG_x_zero, sENG_y_zero

In [5]:
sENG_X, sENG_y = get_sENG(gesture_label=1)

(43177, 4, 14, 1) (43177,)
(43100, 4, 14, 1) (43100,)
(43228, 4, 14, 1) (43228,)
(3555, 8) (3555,)


In [6]:
iENG_X, iENG_y = get_iENG(gesture_label=1)

(4873, 8) (4873,)


In [7]:
sEMG_X, sEMG_y = get_sEMG_3DBs(gesture_label1=9, gesture_label2=10)

(8068, 8) (8068,)


# T-SNE (Gesture 1)

In [8]:
dist = t_sne_compute_dist(sEMG_X, sENG_X, iENG_X, vis=False)
dist

,sEMG,sENG,iENG
sEMG,0.000000,980.912941,980.516776
sENG,980.912941,0.000000,2.419084
iENG,980.516776,2.419084,0.000000


# T-SNE (Gesture 2)

In [9]:
sENG_X, sENG_y = get_sENG(gesture_label=2)
iENG_X, iENG_y = get_iENG(gesture_label=2)
sEMG_X, sEMG_y = get_sEMG_3DBs(gesture_label1=1, gesture_label2=2)

(43177, 4, 14, 1) (43177,)
(43100, 4, 14, 1) (43100,)
(43228, 4, 14, 1) (43228,)
(3555, 8) (3555,)
(5398, 8) (5398,)
(8068, 8) (8068,)


In [10]:
dist = t_sne_compute_dist(sEMG_X, sENG_X, iENG_X, vis=False)
dist

,sEMG,sENG,iENG
sEMG,0.000000,816.312094,814.766907
sENG,816.312094,0.000000,2.837309
iENG,814.766907,2.837309,0.000000


# T-SNE (Gesture 3)

In [11]:
sENG_X, sENG_y = get_sENG(gesture_label=3)
iENG_X, iENG_y = get_iENG(gesture_label=3)
sEMG_X, sEMG_y = get_sEMG_3DBs(gesture_label1=3, gesture_label2=4)

(43177, 4, 14, 1) (43177,)
(43100, 4, 14, 1) (43100,)
(43228, 4, 14, 1) (43228,)
(3555, 8) (3555,)
(4953, 8) (4953,)
(8068, 8) (8068,)


In [12]:
dist = t_sne_compute_dist(sEMG_X, sENG_X, iENG_X, vis=False)
dist

,sEMG,sENG,iENG
sEMG,0.000000,757.534718,755.317974
sENG,757.534718,0.000000,3.325048
iENG,755.317974,3.325048,0.000000


# T-SNE (Gesture 4)

In [13]:
sENG_X, sENG_y = get_sENG(gesture_label=4)
iENG_X, iENG_y = get_iENG(gesture_label=4)
sEMG_X, sEMG_y = get_sEMG_3DBs(gesture_label1=5, gesture_label2=6)

(43177, 4, 14, 1) (43177,)
(43100, 4, 14, 1) (43100,)
(43228, 4, 14, 1) (43228,)
(3555, 8) (3555,)
(4727, 8) (4727,)
(8068, 8) (8068,)


In [14]:
dist = t_sne_compute_dist(sEMG_X, sENG_X, iENG_X, vis=False)
dist

,sEMG,sENG,iENG
sEMG,0.000000,1182.601554,1179.368374
sENG,1182.601554,0.000000,4.142365
iENG,1179.368374,4.142365,0.000000


# T-SNE (Gesture 5)

In [15]:
sENG_X, sENG_y = get_sENG(gesture_label=5)
iENG_X, iENG_y = get_iENG(gesture_label=5)
sEMG_X, sEMG_y = get_sEMG_3DBs(gesture_label1=7, gesture_label2=8)

(43177, 4, 14, 1) (43177,)
(43100, 4, 14, 1) (43100,)
(43228, 4, 14, 1) (43228,)
(3555, 8) (3555,)
(4619, 8) (4619,)
(8068, 8) (8068,)


In [16]:
dist = t_sne_compute_dist(sEMG_X, sENG_X, iENG_X)
dist

,sEMG,sENG,iENG
sEMG,0.000000,1095.994930,1093.682772
sENG,1095.994930,0.000000,2.986904
iENG,1093.682772,2.986904,0.000000
